# Hunyuan3D-2：正面圖批次生成 3D（Kaggle 2×T4，高精度）

跟 Colab 版用同一套流程和檔名規則（見 [README](https://github.com/chousir/colab-ImageTo3D)），差別：

- **兩張 T4 同時做**：每張卡跑一份完整流程（視角 → 形狀 → 貼圖 → 輸出），從同一個 `in/` 各自拿圖，不會重複。
- **高精度**（`QUALITY = "high"`）：所有類別都用形狀網格 512、4K 貼圖（渲染 2048）、較多面數；放不下時自動降一級。
- **Google Drive 用 rclone 同步**：本機路徑和 Colab 一樣是 `/content/drive/MyDrive/RagnarokChange/hunyuan3d`；開始時把 `in/` 抓下來，跑的時候每幾分鐘把 `out/`、`work/` 傳回 Drive，並把完成的圖在 Drive 改名成 `used-…`。
- **模型從 Kaggle Dataset 讀**（用 `kaggle_prepare_models.ipynb` 做一次），不花 GPU 時間下載。

**每次開始前**（右側 Session options）：Accelerator 選 **GPU T4 x2**、**Internet on**；Add Input 加入你的模型 Dataset；Secrets 勾選 `RCLONE_CONF`（和選用的 `HF_TOKEN`）。

**背景執行**：右上 **Save Version → Save & Run All (Commit)**，關掉瀏覽器也會繼續跑（最多 12 小時）；到 `STOP_AFTER_HOURS` 就不再開始新的圖，把成果傳回 Drive 後結束。

**不要同時用 Colab 和 Kaggle 處理同一個 `in/`**。

In [ ]:
# 1. Check GPUs, RAM and disk (Kaggle: Accelerator "GPU T4 x2", Internet on)
import os, time
SESSION_START = time.time()  # Kaggle stops a session 12 hours after it starts.
import platform, shutil, subprocess, sys

print(f"Python: {platform.python_version()}")
# nvidia-smi instead of torch.cuda: this kernel must not hold a CUDA context on GPU 0 for the whole session.
query = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap", "--format=csv,noheader"],
                       capture_output=True, text=True)
if query.returncode != 0:
    raise RuntimeError("No GPU. Session options > Accelerator > GPU T4 x2.")
GPUS = [line.split(", ") for line in query.stdout.strip().splitlines()]
for index, name, memory, capability in GPUS:
    print(f"GPU {index}: {name}  {memory}  sm_{capability.replace('.', '')}")
GPU_COUNT = len(GPUS)
GPU_CAPABILITY = GPUS[0][3]
if GPU_COUNT < 2:
    print("WARNING: one GPU only (Accelerator > GPU T4 x2 gives two); running one worker.")
torch_version = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__, torch.version.cuda)"],
                               capture_output=True, text=True).stdout.strip()
print(f"Torch : {torch_version}")
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
print(f"RAM   : {ram_gb:.1f} GB")
print(f"Disk  : {shutil.disk_usage('/').free / 1024**3:.0f} GB free (/), "
      f"{shutil.disk_usage('/kaggle/working').free / 1024**3:.0f} GB free (/kaggle/working)")
nvcc = shutil.which("nvcc")
print(subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout.strip().splitlines()[-1] if nvcc else "nvcc: not found")
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise RuntimeError("No internet. Session options > Internet on (needs a phone-verified account).")

In [ ]:
# 2. Settings, Google Drive (rclone) and model dataset
# Drive folder: same path as the Colab mount, kept in step with Google Drive by rclone.
DRIVE_ROOT = "/content/drive/MyDrive/RagnarokChange/hunyuan3d"
REMOTE = "gdrive:RagnarokChange/hunyuan3d"  # <rclone remote name>:<folder under My Drive>
# high: octree 512, 4K textures and more faces for every category. standard: the Colab settings.
QUALITY = "high"  # "high" or "standard"
HEAD_MODE = "bald"  # "bald" or "keep"
VIEW_COUNT = 4  # 4 or 6
VIEW_SEED = 1234
SHAPE_MODEL = "hunyuan3d-dit-v2-mv"  # or "hunyuan3d-dit-v2-mv-turbo"
SINGLE_SHAPE_MODEL = "hunyuan3d-dit-v2-0"  # or "hunyuan3d-dit-v2-0-turbo"
SHAPE_SEED = 1234
PAINT_MODEL = "hunyuan3d-paint-v2-0"  # or "hunyuan3d-paint-v2-0-turbo"
# Fallback texture when a category's 4K texture does not fit in GPU memory.
TEXTURE_SIZE = 2048
WATCH = True
POLL_MINUTES = 2
IDLE_STOP_MINUTES = 30
SYNC_MINUTES = 5  # how often results go to Drive and new images come from Drive
# No new image starts after this many hours, leaving time to finish and upload before Kaggle's 12-hour limit.
STOP_AFTER_HOURS = 11
CLAIM_BATCH = 2  # images a GPU takes at a time
# Without an attached model dataset, download from Hugging Face in this session (uses GPU time).
ALLOW_HF_DOWNLOAD = True
PIPELINE_REF = "claude/gallant-thompson-py38mf"
RCLONE_VERSION = "v1.70.3"

if "SESSION_START" not in globals():
    raise RuntimeError("Run block 1 first.")
import os, shutil, subprocess, sys, zipfile, urllib.request
from kaggle_secrets import UserSecretsClient

def secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    result = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

# Pipeline code: always refresh so fixes pushed to the branch apply on the next run.
PIPELINE_DIR = "/content/colab-ImageTo3D"
CONFIG_PATH = "/content/imageto3d_config.json"
os.makedirs("/content", exist_ok=True)
if not os.path.exists(PIPELINE_DIR):
    sh(f"git clone https://github.com/chousir/colab-ImageTo3D.git {PIPELINE_DIR}")
sh(f"git fetch -q origin {PIPELINE_REF} && git checkout -q -B run FETCH_HEAD", cwd=PIPELINE_DIR)
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)
import importlib
import imageto3d.config, imageto3d.assets, imageto3d.models, imageto3d.drive_sync, imageto3d.run
for module in (imageto3d.config, imageto3d.assets, imageto3d.models, imageto3d.drive_sync, imageto3d.run):
    importlib.reload(module)

# rclone, configured from the RCLONE_CONF secret (contents of your rclone.conf).
if not shutil.which("rclone"):
    archive = f"/tmp/rclone-{RCLONE_VERSION}.zip"
    urllib.request.urlretrieve(f"https://github.com/rclone/rclone/releases/download/{RCLONE_VERSION}/"
                               f"rclone-{RCLONE_VERSION}-linux-amd64.zip", archive)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extract(f"rclone-{RCLONE_VERSION}-linux-amd64/rclone", "/tmp")
    shutil.copy(f"/tmp/rclone-{RCLONE_VERSION}-linux-amd64/rclone", "/usr/local/bin/rclone")
    os.chmod("/usr/local/bin/rclone", 0o755)
rclone_conf = secret("RCLONE_CONF")
if not rclone_conf:
    raise RuntimeError("Secret RCLONE_CONF is missing: Add-ons > Secrets, paste your rclone.conf, attach it to this notebook.")
os.makedirs(os.path.expanduser("~/.config/rclone"), exist_ok=True)
with open(os.path.expanduser("~/.config/rclone/rclone.conf"), "w") as file:
    file.write(rclone_conf)
sh(f"rclone mkdir '{REMOTE}/in'")
SYNC = imageto3d.drive_sync.DriveSync(REMOTE, DRIVE_ROOT)

for name in ("in", "out", "work", "wheels"):
    os.makedirs(os.path.join(DRIVE_ROOT, name), exist_ok=True)
SYNC.pull_inputs()
SYNC.pull_folder("wheels")

# Model weights: the attached Kaggle dataset, else a download into this session.
MODEL_ROOT = imageto3d.models.find_root("/kaggle/input")
if MODEL_ROOT:
    print("Model dataset:", MODEL_ROOT)
elif ALLOW_HF_DOWNLOAD:
    print("WARNING: no model dataset attached (Add Input). Downloading from Hugging Face; this uses GPU time.")
    MODEL_ROOT = "/content/models"
    if not os.path.exists(os.path.join(MODEL_ROOT, imageto3d.models.MANIFEST)):
        imageto3d.models.download(MODEL_ROOT, token=secret("HF_TOKEN"))
else:
    raise RuntimeError("No model dataset under /kaggle/input. Add Input > your dataset, or set ALLOW_HF_DOWNLOAD.")
HF_HOME = "/content/hf_cache"  # anything not in the dataset is downloaded here
REMBG_HOME = "/content/rembg_cache"
WHEEL_CACHE = os.path.join(DRIVE_ROOT, "wheels")
os.makedirs(REMBG_HOME, exist_ok=True)
rembg_model = os.path.join(MODEL_ROOT, "rembg", "u2net.onnx")
if os.path.exists(rembg_model) and not os.path.exists(os.path.join(REMBG_HOME, "u2net.onnx")):
    shutil.copy(rembg_model, REMBG_HOME)
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")

SETTINGS = dict(
    drive_root=DRIVE_ROOT, hf_home=HF_HOME, rembg_home=REMBG_HOME, model_root=MODEL_ROOT,
    head_mode=HEAD_MODE, view_count=VIEW_COUNT, view_seed=VIEW_SEED,
    shape_model=SHAPE_MODEL, single_shape_model=SINGLE_SHAPE_MODEL, shape_seed=SHAPE_SEED, paint_model=PAINT_MODEL,
    texture_size=TEXTURE_SIZE, poll_seconds=POLL_MINUTES * 60, idle_stop_minutes=IDLE_STOP_MINUTES,
    deadline=SESSION_START + STOP_AFTER_HOURS * 3600, claim_batch=CLAIM_BATCH,
)
if QUALITY == "high":
    SETTINGS["presets"] = imageto3d.config.HIGH_QUALITY_PRESETS

# Intermediate files of pending images, so work from an earlier session resumes where it stopped.
probe = imageto3d.config.Config(**SETTINGS)
pending = [asset.key for asset in imageto3d.assets.pending(probe)]
SYNC.pull_work(pending)
imageto3d.assets.clear_claims(probe)
print(f"{len(pending)} image(s) waiting:", ", ".join(pending) or "none")
print("Settings ready.")

In [ ]:
# 3. Install (do not stop it while it runs)
# Installs pinned packages and reuses CUDA extensions cached in Drive/wheels. No restart needed:
# the heavy work runs in separate processes that start after the install.
if "SETTINGS" not in globals():
    raise RuntimeError("Settings are missing. Run blocks 1 and 2 first.")
import glob, os, shutil, subprocess, sys, sysconfig

REPO = "/content/Hunyuan3D-2"
MV_REPO = "/content/MV-Adapter"
HUNYUAN_COMMIT = "f8db63096c8282cb27354314d896feba5ba6ff8a"
MV_COMMIT = "4277e0018232bac82bb2c103caf0893cedb711be"
MARK = "/content/.imageto3d_installed_v3"
CONSTRAINTS = "/content/constraints.txt"
# Same pins as the Colab notebook; Kaggle's own torch, numpy and opencv are kept through the constraints file.
PINNED = [
    "transformers==4.57.3", "diffusers==0.34.0", "huggingface_hub==0.36.0", "accelerate==1.10.1",
    "peft==0.17.1", "omegaconf==2.3.0", "einops==0.8.1", "pymeshlab==2025.7.post1", "xatlas==0.0.11",
    "pygltflib==1.16.5", "trimesh==4.8.3", "onnxruntime==1.22.1", "pybind11==3.0.1", "ninja==1.13.0",
    "pooch==1.8.2", "jsonschema==4.25.1",
    # MV-Adapter imports these at module load (mvadapter/utils/typing.py).
    "jaxtyping==0.3.2", "typeguard==4.4.4",
]
PINNED_NO_DEPS = ["rembg==2.0.67", "pymatting==1.1.14"]
NVDIFFRAST = "git+https://github.com/NVlabs/nvdiffrast.git@v0.3.3"


def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    process = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in process.stdout:
        tail = (tail + [line])[-40:]
        if not line.startswith(("  ", "Requirement already")):
            print(line, end="", flush=True)
    if process.wait() != 0:
        print("".join(tail))
        raise RuntimeError(f"Command failed: {cmd}")


if not os.path.exists(MARK):
    import torch
    capability = tuple(int(part) for part in GPU_CAPABILITY.split("."))  # from nvidia-smi, no CUDA context here
    abi_tag = f"py{sys.version_info[0]}{sys.version_info[1]}-torch{torch.__version__.replace('+', '_')}-sm{capability[0]}{capability[1]}"
    wheel_dir = os.path.join(WHEEL_CACHE, abi_tag)
    os.makedirs(wheel_dir, exist_ok=True)

    # Keep Colab's compiled stack; a different torch or numpy would break CUDA extensions.
    frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    keep = ("numpy==", "scipy==", "torch==", "torchvision==", "triton==", "numba==", "llvmlite==",
            "pillow==", "opencv-python==", "opencv-python-headless==", "scikit-image==")
    with open(CONSTRAINTS, "w") as file:
        file.write("".join(line + "\n" for line in frozen.splitlines() if line.lower().startswith(keep)))

    if not os.path.exists(REPO):
        sh(f"git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git {REPO}")
    sh(f"git checkout -q {HUNYUAN_COMMIT}", cwd=REPO)
    if not os.path.exists(MV_REPO):
        sh(f"git clone https://github.com/huanngzh/MV-Adapter.git {MV_REPO}")
    sh(f"git checkout -q {MV_COMMIT}", cwd=MV_REPO)

    sh(f"pip install -c {CONSTRAINTS} " + " ".join(f"'{package}'" for package in PINNED))
    sh(f"pip install -c {CONSTRAINTS} --no-deps " + " ".join(PINNED_NO_DEPS))
    sh(f"pip install --no-deps -e {REPO}")
    # nvdiffrast 0.3.x compiles lazily and is only imported by MV-Adapter, so this install is quick.
    sh(f"pip install --no-build-isolation --no-deps {NVDIFFRAST}")

    # custom_rasterizer: build once per Python/torch/GPU combination, then reuse the wheel from Drive.
    wheels = sorted(glob.glob(os.path.join(wheel_dir, "custom_rasterizer-*.whl")))
    if not wheels:
        os.environ["TORCH_CUDA_ARCH_LIST"] = f"{capability[0]}.{capability[1]}"
        build_dir = "/content/wheel_build"
        shutil.rmtree(build_dir, ignore_errors=True)
        sh(f"pip wheel -c {CONSTRAINTS} --no-build-isolation --no-deps -w {build_dir} .",
           cwd=f"{REPO}/hy3dgen/texgen/custom_rasterizer")
        for wheel in glob.glob(os.path.join(build_dir, "custom_rasterizer-*.whl")):
            shutil.copy2(wheel, wheel_dir)
        wheels = sorted(glob.glob(os.path.join(wheel_dir, "custom_rasterizer-*.whl")))
    else:
        print("Using cached wheel:", wheels[-1])
    sh(f"pip install --no-deps --force-reinstall '{wheels[-1]}'")

    renderer_dir = f"{REPO}/hy3dgen/texgen/differentiable_renderer"
    mesh_processor = f"mesh_processor{sysconfig.get_config_var('EXT_SUFFIX')}"
    cached_processor = os.path.join(wheel_dir, mesh_processor)
    if not os.path.exists(cached_processor):
        try:
            sh(f"c++ -O3 -Wall -shared -std=c++11 -fPIC $({sys.executable} -m pybind11 --includes) "
               f"mesh_processor.cpp -o {mesh_processor}", cwd=renderer_dir)
            shutil.copy2(os.path.join(renderer_dir, mesh_processor), cached_processor)
        except RuntimeError:
            print("mesh_processor build failed; using the slower Python fallback.")
    else:
        print("Using cached", cached_processor)
        shutil.copy2(cached_processor, renderer_dir)
    open(MARK, "w").close()
    SYNC.push()  # compiled extensions go to Drive/wheels for the next session

# Check every import in a fresh process, exactly as the stage processes will load them.
check = ("import sys; sys.path[:0] = [%r, %r, %r]; import torch, custom_rasterizer, hy3dgen.shapegen, hy3dgen.texgen, rembg; "
         "from mvadapter.pipelines.pipeline_mvadapter_i2mv_sdxl import MVAdapterI2MVSDXLPipeline; "
         "from scripts.inference_i2mv_sdxl import run_pipeline; import imageto3d.run; "
         "print('torch', torch.__version__, 'pipelines OK')" % (PIPELINE_DIR, MV_REPO, REPO))
result = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
print(result.stdout, result.stderr[-3000:])
if result.returncode != 0:
    os.remove(MARK)
    raise RuntimeError("Pipeline import check failed. Send the last 30 lines above for diagnosis.")

if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)
import importlib
import imageto3d.config, imageto3d.assets, imageto3d.run
for module in (imageto3d.config, imageto3d.assets, imageto3d.run):
    importlib.reload(module)
imageto3d.config.Config(**SETTINGS, hunyuan_repo=REPO, mv_repo=MV_REPO).save(CONFIG_PATH)
print("Install OK. Config written to", CONFIG_PATH)

In [ ]:
# 4. Generate 3D for every image in Drive/in, one worker per GPU
# Each worker takes CLAIM_BATCH images at a time and runs views, shape, paint and export on its own GPU.
# This cell copies results to Drive every SYNC_MINUTES and fetches new images; stopping it uploads what is done.
if "CONFIG_PATH" not in globals() or not os.path.exists(CONFIG_PATH):
    raise RuntimeError("Run blocks 1, 2 and 3 first.")
import subprocess, sys, threading, time
import imageto3d.assets, imageto3d.config

config = imageto3d.config.load(CONFIG_PATH)
imageto3d.assets.clear_claims(config)

def stream(index, process):
    for line in process.stdout:
        print(f"[gpu{index}] {line}", end="", flush=True)

def sync_now():
    try:
        SYNC.push()
        SYNC.pull_inputs()
        SYNC.pull_work([asset.key for asset in imageto3d.assets.pending(config)
                        if not os.path.exists(asset.work)])
    except Exception as error:  # Drive hiccups must not stop generation; the next sync retries.
        print(f"[sync] {error}", flush=True)

workers = []
for index in range(GPU_COUNT):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(index), PYTHONPATH=PIPELINE_DIR)
    command = [sys.executable, "-u", "-m", "imageto3d.run", "--config", CONFIG_PATH, "--worker", str(index)]
    process = subprocess.Popen(command + ([] if WATCH else ["--once"]), env=env, cwd=PIPELINE_DIR, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    threading.Thread(target=stream, args=(index, process), daemon=True).start()
    workers.append(process)
    time.sleep(20)  # stagger model loading so both workers do not read the same weights at once

try:
    last_sync = time.time()
    while any(process.poll() is None for process in workers):
        time.sleep(15)
        if time.time() - last_sync >= SYNC_MINUTES * 60:
            sync_now()
            last_sync = time.time()
except KeyboardInterrupt:
    print("Stopping workers...", flush=True)
    for process in workers:
        process.terminate()
    for process in workers:
        process.wait()
finally:
    print("Uploading results to Drive...", flush=True)
    sync_now()
    done = sorted(name for name in os.listdir(config.out_dir) if name.endswith(".glb"))
    print(f"Done. {len(done)} GLB file(s) in {REMOTE}/out")

In [ ]:
# 5. Review results (optional)
# COPY_TO_OUTPUT also copies the GLB files to /kaggle/working (the notebook's Output tab).
COPY_TO_OUTPUT = False
import glob, os, shutil, time
from IPython.display import Image as ShowImage, display
import imageto3d.config

config = imageto3d.config.load(CONFIG_PATH)
outputs = sorted(glob.glob(os.path.join(config.out_dir, "*.glb")), key=os.path.getmtime)
for glb in outputs[-10:]:
    print(f"{os.path.basename(glb):55s} {os.path.getsize(glb) / 1024**2:5.1f} MB")
    sheet = glb[:-4] + "-views.jpg"
    if os.path.exists(sheet):
        display(ShowImage(filename=sheet, width=768))
for status_path in glob.glob(os.path.join(config.work_dir, "*", "status.json")):
    status = open(status_path).read()
    if '"last_error": "' in status:
        print("Problem:", os.path.basename(os.path.dirname(status_path)), status)
if COPY_TO_OUTPUT and outputs:
    shutil.copytree(config.out_dir, "/kaggle/working/out", dirs_exist_ok=True)
    print("Copied to /kaggle/working/out")

## 出問題時（Kaggle）

| 狀況 | 做法 |
|---|---|
| 區塊 1：No GPU／只有一張 GPU | Session options → Accelerator 選 **GPU T4 x2** |
| 區塊 1：No internet | Session options → Internet on（帳號要先做手機驗證） |
| 區塊 2：Secret RCLONE_CONF is missing | Add-ons → Secrets 新增 `RCLONE_CONF`（貼上 rclone.conf 全文），並勾選給這個 notebook 用 |
| 區塊 2：rclone 失敗（token expired、invalid_grant） | 在自己電腦跑 `rclone config reconnect gdrive:`，再把新的 rclone.conf 貼回 Secret |
| 區塊 2：no model dataset attached | Add Input → Your Work → Datasets 加入用 `kaggle_prepare_models.ipynb` 做的資料集 |
| 區塊 3 安裝失敗 | 看最後 30 行錯誤；Kaggle 映像更新後套件不合時，notebook 設定 Environment 選「Pin to original environment」 |
| `[gpu0] … out of GPU memory … retrying` | 正常：該張圖的 512 網格或 4K 貼圖放不下，自動降一級 |
| 常出現 killed: out of memory | `CLAIM_BATCH = 1`，或 `QUALITY = "standard"` |
| 12 小時快到 | 到 `STOP_AFTER_HOURS` 就不再開新圖；沒做完的圖下次 session 從中斷的階段接著做 |
| 其他（SKIPPED、WAITING、檔名、方向） | 同 Colab 版的「出問題時」 |